## Retriever And Chain With Langchain


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("attention.pdf")
docs = loader.load()
docs

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
text_splitter.split_documents(docs)[:5]

In [ ]:
documents=text_splitter.split_documents(docs)
documents

In [ ]:
## FAISS Vector Database

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

db=FAISS.from_documents(documents[:30],embeddings)

In [ ]:
query="Who are the authors of the attention is all you need research paper"
result=db.similarity_search(query)
result[0].page_content

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains.question_answering import load_qa_chain

prompt_template = """ 
Answer the following question based only on the provided context.Think step by step before providing a detailed answer.I will tip you $1000 if the user finds the answer helpful.
Context:\n {context}?\n
Question: \n{question}\n

Answer :

"""
model = ChatGoogleGenerativeAI(model="gemini-pro",temperature=0.3)

prompt = PromptTemplate(template = prompt_template, input_variables = ["context", "question"])
chain = load_qa_chain(model, chain_type="stuff", prompt=prompt)


In [ ]:
"""
Retrievers: A retriever is an interface that returns documents given
 an unstructured query. It is more general than a vector store.
 A retriever does not need to be able to store documents, only to 
 return (or retrieve) them. Vector stores can be used as the backbone
 of a retriever, but there are other types of retrievers as well. 
 https://python.langchain.com/docs/modules/data_connection/retrievers/   
"""

retriever=db.as_retriever()
retriever

In [ ]:
response = chain.invoke(
        {"input_documents":docs, "question": "An attention function can be described as mapping a query"}
        , return_only_outputs=True)

In [ ]:
response['output_text']